In [1]:
import mlflow 
mlflow.autolog(silent=True)

import mlflow
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [2]:
mlflow.set_experiment("Reco Agricole")
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))

In [3]:
# ── Palette ────────────────────────────────────────────────────────────────────
C_A     = "#F97066"   # coral-red  → Dummy Regressor / only_real
C_B     = "#6C63FF"   # violet     → XGBoost Fine-Tuned / base (augmented)
C_BG    = "#FFFFFF"
C_PANEL = "#F7F8FC"
C_GRID  = "#E5E7EF"
C_TEXT  = "#1A1D2E"
C_SUB   = "#6B7280"
 
CROPS   = ["Wheat", "Maize", "Rice"]
METRICS = [
    {"key": "MAE", "note": "Mean Absolute Error  ↓ lower is better"},
    {"key": "MAPE", "note": "Mean Absolute Percentage Error  ↓ lower is better"},
]

In [4]:
def make_comparison_figure(data_a, data_b, label_a, label_b, figure_title, out_stem):
    """
    Create and save a side-by-side bar chart (RMSE | MAPE) comparing
    two groups (A vs B) across CROPS.
 
    Parameters
    ----------
    data_a, data_b : dict  {crop: {metric: value}}
    label_a, label_b : str  Legend labels
    figure_title : str      Suptitle
    out_stem : str          Output file base name (no extension)
    """
 
    BAR_W = 0.32
    x     = np.arange(len(CROPS))
 
    plt.rcParams.update({
        "font.family":        "DejaVu Sans",
        "axes.spines.top":    False,
        "axes.spines.right":  False,
        "axes.spines.left":   False,
        "axes.spines.bottom": False,
        "xtick.bottom":       False,
        "ytick.left":         False,
        "figure.dpi":         300,
    })
 
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), facecolor=C_BG)
    fig.subplots_adjust(wspace=0.38, left=0.07, right=0.97, top=0.82, bottom=0.15)
 
    for ax, m in zip(axes, METRICS):
        key = m["key"]
 
        vals_a = [data_a[c][key] for c in CROPS]
        vals_b = [data_b[c][key] for c in CROPS]
 
        # Panel background
        ax.set_facecolor(C_PANEL)
        for spine in ax.spines.values():
            spine.set_visible(False)
 
        # Bars
        bars_a = ax.bar(x - BAR_W / 2, vals_a, width=BAR_W,
                        color=C_A, zorder=3, linewidth=0)
        bars_b = ax.bar(x + BAR_W / 2, vals_b, width=BAR_W,
                        color=C_B, zorder=3, linewidth=0)
 
        # Value labels above bars
        max_val = max(*vals_a, *vals_b)
        offset  = max_val * 0.025
 
        for bars, vals in [(bars_a, vals_a), (bars_b, vals_b)]:
            for bar, val in zip(bars, vals):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + offset,
                    f"{val:.3g}",
                    ha="center", va="bottom",
                    fontsize=8.5, color=C_TEXT, fontweight="600",
                )
 
        # Horizontal grid only
        ax.set_axisbelow(True)
        ax.yaxis.grid(True, color=C_GRID, linewidth=1.2, zorder=0)
 
        ax.set_xticks(x)
        ax.set_xticklabels(CROPS, fontsize=11, color=C_TEXT, fontweight="600")
        ax.tick_params(axis="y", labelsize=9, colors=C_SUB)
        ax.set_ylim(0, max_val * 1.30)
 
        # Axis titles
        ax.set_title(key, fontsize=15, fontweight="700",
                     color=C_TEXT, pad=14, loc="center")
        ax.text(0, 1.00, m["note"], transform=ax.transAxes,
                fontsize=8.5, color=C_SUB, va="bottom")
 
    # Figure-level title & legend
    fig.suptitle(figure_title, fontsize=14, fontweight="700",
                 color=C_TEXT, x=0.52, y=0.97)
 
    legend_handles = [
        mpatches.Patch(facecolor=C_A, label=label_a, linewidth=0),
        mpatches.Patch(facecolor=C_B, label=label_b, linewidth=0),
    ]
    fig.legend(
        handles=legend_handles,
        loc="upper center", bbox_to_anchor=(0.52, 0.91),
        ncol=2, frameon=False, fontsize=10, labelcolor=C_TEXT,
        handlelength=1.4, handletextpad=0.5, columnspacing=1.5,
    )
 
    # Save PNG + PDF
    ext = "png"
    path = f"{out_stem}.{ext}"
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor=C_BG)
    print(f"Saved → {path}")
 
    plt.close(fig)
 

# Baseline vs Fine-tuned model

In [5]:
dummy_regressor = {
    "Wheat": {"RMSE": 1.851174, "MAPE": 0.722025, "MAE": 1.41983939952637},
    "Maize": {"RMSE": 2.711227, "MAPE": 1.065836, "MAE": 2.15969675492792},
    "Rice":  {"RMSE": 1.948146, "MAPE": 0.486125, "MAE": 1.53580105726632},
}
xgboost_final = {
    "Wheat": {"RMSE": 0.22599680248991066,  "MAPE": 0.053616114084093905, "MAE": 0.11500644913392163},
    "Maize": {"RMSE": 0.32494536303636196,  "MAPE": 0.07147760084248363, "MAE": 0.15560211423198983},
    "Rice":  {"RMSE": 0.28291433738444843,  "MAPE": 0.045420693611175074, "MAE": 0.14164011457268025},
}

make_comparison_figure(
    data_a=dummy_regressor,
    data_b=xgboost_final,
    label_a="Dummy Regressor",
    label_b="XGBoost Fine-Tuned",
    figure_title="Crop Yield Prediction - Model Performance Comparison",
    out_stem="../assets/fig1_model_comparison",
)

Saved → ../assets/fig1_model_comparison.png


# Real vs Augmented Real data

In [6]:
only_real = {
    "Wheat": {"RMSE": 0.3116583776413684, "MAPE": 0.07104195968367448, "MAE": 0.15808196041636208},
    "Maize": {"RMSE": 0.4874201919712619, "MAPE": 0.09749574153458655, "MAE": 0.21986171326341025},
    "Rice":  {"RMSE": 0.3117487460052828, "MAPE": 0.053331002191360645, "MAE": 0.15641188880874884},
}

make_comparison_figure(
    data_a=only_real,
    data_b=xgboost_final,
    label_a="Real data only",
    label_b="Augmented data",
    figure_title="Crop Yield Prediction - Dataset Comparison",
    out_stem="../assets/fig2_dataset_comparison",
)

Saved → ../assets/fig2_dataset_comparison.png


# Feature importance

In [7]:
import pandas as pd
knn_df = pd.read_csv("../data/processed/knn_merged.csv")

In [8]:
import xgboost as xgb
import pandas as pd
import numpy as np
import shap
from pathlib import Path
from tqdm.notebook import tqdm

# Re-use constants from models_store
VALID_CROPS = {"Wheat", "Rice", "Maize"}

COUNTRIES = [
    'Algeria', 'Angola', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan',
    'Bahamas', 'Bangladesh', 'Belarus', 'Belgium', 'Botswana', 'Brazil', 'Bulgaria',
    'Burkina Faso', 'Burundi', 'Cameroon', 'Canada', 'Central African Republic', 'Chile',
    'Colombia', 'Croatia', 'Denmark', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador',
    'Eritrea', 'Estonia', 'Finland', 'France', 'Germany', 'Ghana', 'Greece', 'Guatemala',
    'Guinea', 'Guyana', 'Haiti', 'Honduras', 'Hungary', 'India', 'Indonesia', 'Iraq', 'Ireland',
    'Italy', 'Jamaica', 'Japan', 'Kazakhstan', 'Kenya', 'Latvia', 'Lebanon', 'Lesotho', 'Libya',
    'Lithuania', 'Madagascar', 'Malawi', 'Malaysia', 'Mali', 'Mauritania', 'Mauritius', 'Mexico',
    'Montenegro', 'Morocco', 'Mozambique', 'Namibia', 'Nepal', 'Netherlands', 'New Zealand',
    'Nicaragua', 'Niger', 'Norway', 'Pakistan', 'Papua New Guinea', 'Peru', 'Poland', 'Portugal',
    'Qatar', 'Romania', 'Rwanda', 'Saudi Arabia', 'Senegal', 'Slovenia', 'South Africa', 'Spain',
    'Sri Lanka', 'Sudan', 'Suriname', 'Sweden', 'Switzerland', 'Tajikistan', 'Thailand', 'Tunisia',
    'Turkey', 'Uganda', 'Ukraine', 'United Kingdom', 'Uruguay', 'Zambia', 'Zimbabwe',
]

SOIL_TYPES = ['Clay', 'Loam', 'Peaty', 'Sandy', 'Silt']

FIELD_TYPES: dict[str, type | tuple] = {
    "rain (mm)":         (int, float),
    "temp (C)":          (int, float),
    "Year":              int,
    "pesticides_tonnes": (int, float),
    "Area":              str,
    "Days_to_Harvest":   int,
    "Irrigation_Used":   bool,
    "Fertilizer_Used":   bool,
    "Soil_Type":         str,
}

FEATURE_GROUPS = {
    'rain (mm)':         ['rain (mm)'],
    'temp (C)':          ['temp (C)'],
    'Year':              ['Year'],
    'pesticides_tonnes': ['pesticides_tonnes'],
    'Days_to_Harvest':   ['Days_to_Harvest'],
    'Irrigation_Used':   ['Irrigation_Used'],
    'Fertilizer_Used':   ['Fertilizer_Used'],
    'Soil_Type':         [f'Soil_Type_{s}' for s in SOIL_TYPES],
    'Area':              [f'Area_{c}' for c in COUNTRIES],
}

MODELS_PATH = {
    "Maize": Path("../src/models/maize_model.json"),
    "Rice":  Path("../src/models/rice_model.json"),
    "Wheat": Path("../src/models/wheat_model.json"),
}

def compute_global_shap(df_ohe: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Compute global SHAP importance (mean |SHAP|) for each of the three crop models.

    Parameters
    ----------
    df_ohe : pd.DataFrame
        Pre-transformed (one-hot-encoded) dataset matching the feature order
        produced by Models.transform_data(). Shape: (n_samples, n_features).

    Returns
    -------
    dict[str, pd.DataFrame]
        Keys are crop names ("Maize", "Rice", "Wheat").
        Each value is a DataFrame with columns:
            - "feature"         : original feature name (pre-OHE)
            - "mean_abs_shap"   : mean absolute SHAP value across all samples
        Sorted by importance descending.
    """
    results = {}
    ohe_feature_names = list(df_ohe.drop(["item", "yield (t/ha)"], axis=1).columns)

    for crop, model_path in tqdm(MODELS_PATH.items()):
        # ── Load model ────────────────────────────────────────────────────────
        model = xgb.Booster()
        model.load_model(model_path)

        df = df_ohe[df_ohe["item"] == crop].drop(["item", "yield (t/ha)"], axis=1)

        # ── Raw SHAP matrix  (n_samples × n_ohe_features) ────────────────────
        explainer   = shap.TreeExplainer(model=model)
        shap_matrix = explainer.shap_values(df)   # shape: (n_samples, n_ohe_features)

        # ── Aggregate OHE columns → original features ────────────────────────
        aggregated_rows = []
        for original_feat, ohe_cols in FEATURE_GROUPS.items():
            indices = [
                ohe_feature_names.index(c)
                for c in ohe_cols
                if c in ohe_feature_names
            ]
            # Sum SHAP across OHE columns, then take mean |SHAP| across samples
            grouped_shap    = shap_matrix[:, indices].sum(axis=1)   # (n_samples,)
            mean_abs        = float(np.mean(np.abs(grouped_shap)))
            aggregated_rows.append({"feature": original_feat, "mean_abs_shap": mean_abs})

        importance_df = (
            pd.DataFrame(aggregated_rows)
              .sort_values("mean_abs_shap", ascending=False)
              .reset_index(drop=True)
        )
        results[crop] = importance_df
        print(f"[{crop}] global SHAP computed over {len(df)} samples.")

    return results

In [9]:
global_shap = compute_global_shap(knn_df)

  0%|          | 0/3 [00:00<?, ?it/s]

[Maize] global SHAP computed over 4121 samples.
[Rice] global SHAP computed over 3388 samples.
[Wheat] global SHAP computed over 3857 samples.


In [10]:
print(global_shap)

{'Maize':              feature  mean_abs_shap
0    Fertilizer_Used       1.557100
1    Irrigation_Used       0.596465
2          rain (mm)       0.448094
3               Year       0.271755
4  pesticides_tonnes       0.244924
5               Area       0.181986
6           temp (C)       0.166958
7    Days_to_Harvest       0.069118
8          Soil_Type       0.065948, 'Rice':              feature  mean_abs_shap
0    Irrigation_Used       0.925470
1    Fertilizer_Used       0.486387
2          rain (mm)       0.390358
3  pesticides_tonnes       0.316819
4               Year       0.199416
5               Area       0.188651
6          Soil_Type       0.138100
7    Days_to_Harvest       0.093827
8           temp (C)       0.067346, 'Wheat':              feature  mean_abs_shap
0    Fertilizer_Used       0.745687
1          rain (mm)       0.483067
2    Irrigation_Used       0.289252
3           temp (C)       0.270962
4               Area       0.200548
5  pesticides_tonnes       0.120372

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

global_shap = {
    'Maize': {'Fertilizer_Used': 1.5571, 'Irrigation_Used': 0.5965, 'rain (mm)': 0.4481,
              'Year': 0.2718, 'pesticides_tonnes': 0.2449, 'Area': 0.182,
              'temp (C)': 0.167, 'Days_to_Harvest': 0.0691, 'Soil_Type': 0.0659},
    'Rice':  {'Irrigation_Used': 0.9255, 'Fertilizer_Used': 0.4864, 'rain (mm)': 0.3904,
              'pesticides_tonnes': 0.3168, 'Year': 0.1994, 'Area': 0.1887,
              'Soil_Type': 0.138, 'Days_to_Harvest': 0.0938, 'temp (C)': 0.0673},
    'Wheat': {'Fertilizer_Used': 0.7457, 'rain (mm)': 0.4831, 'Irrigation_Used': 0.2893,
              'temp (C)': 0.271, 'Area': 0.2005, 'pesticides_tonnes': 0.1204,
              'Year': 0.1171, 'Days_to_Harvest': 0.0467, 'Soil_Type': 0.0425}
}

features = ['Fertilizer_Used', 'Irrigation_Used', 'rain (mm)', 'Year',
            'pesticides_tonnes', 'Area', 'temp (C)', 'Days_to_Harvest', 'Soil_Type']

labels = ['Fertilizer used', 'Irrigation used', 'Rainfall (mm)', 'Year',
          'Pesticides (t)', 'Area', 'Temperature (°C)', 'Days to harvest', 'Soil type']

colors = {'Maize': '#EF9F27', 'Rice': '#1D9E75', 'Wheat': '#378ADD'}
crops   = list(colors.keys())

# --- Sort features by average SHAP (descending) ---
avg_shap = {f: np.mean([global_shap[c][f] for c in crops]) for f in features}
sorted_pairs   = sorted(zip(features, labels), key=lambda x: avg_shap[x[0]])
sorted_features, sorted_labels = zip(*sorted_pairs)

# --- Layout ---
n_features  = len(sorted_features)
n_crops     = len(crops)
bar_height  = 0.22
group_gap   = 0.08
group_size  = n_crops * bar_height + group_gap
y_positions = np.arange(n_features) * group_size

# --- Style ---
plt.rcParams.update({
    'font.family':      'sans-serif',
    'axes.facecolor':   '#ffffff',
    'figure.facecolor': '#ffffff',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.spines.left': False,
    'axes.spines.bottom': True,
    'axes.edgecolor':   '#cccbc5',
    'axes.linewidth':   0.6,
    'xtick.color':      '#888780',
    'ytick.color':      '#444441',
    'grid.color':       '#e8e7e1',
    'grid.linewidth':   0.6,
})

fig, ax = plt.subplots(figsize=(9, 6))
fig.subplots_adjust(left=0.2, right=0.88, top=0.88, bottom=0.1)

for i, (feat, label) in enumerate(zip(sorted_features, sorted_labels)):
    for j, crop in enumerate(crops):
        y     = y_positions[i] + j * bar_height
        value = global_shap[crop][feat]
        bar   = ax.barh(y, value, height=bar_height * 0.85,
                        color=colors[crop], linewidth=0)
        # Value label
        ax.text(value + 0.015, y, f'{value:.3f}',
                va='center', ha='left', fontsize=8.5,
                color='#666560')

# --- Y-axis ticks centred on each group ---
group_centers = y_positions + (n_crops - 1) * bar_height / 2
ax.set_yticks(group_centers)
ax.set_yticklabels(sorted_labels, fontsize=11, color='#444441')

ax.set_xlabel('Mean |SHAP| value', fontsize=11, color='#888780', labelpad=8)
ax.xaxis.grid(True, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(axis='x', colors='#888780', labelsize=9.5)
ax.tick_params(axis='y', length=0)

# --- Title ---
ax.set_title('Feature importance by crop (Mean |SHAP|)',
             fontsize=13, color='#2C2C2A', fontweight='normal',
             loc='left', pad=14)

# --- Legend ---
patches = [mpatches.Patch(color=colors[c], label=c) for c in crops]
ax.legend(handles=patches, loc='lower right', fontsize=10,
          frameon=False, labelcolor='#666560',
          handlelength=1.2, handleheight=0.9)

plt.savefig('../assets/fig3_shapley.png', dpi=150, bbox_inches='tight',
            facecolor='#ffffff')

In [12]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── Feature metadata ──────────────────────────────────────────────────────────
FEATURE_TYPES = {
    'rain (mm)':         'continuous',
    'temp (C)':          'continuous',
    'Year':              'continuous',
    'pesticides_tonnes': 'continuous',
    'Days_to_Harvest':   'continuous',
    'Irrigation_Used':   'boolean',
    'Fertilizer_Used':   'boolean',
    'Soil_Type':         'categorical',
    'Area':              'categorical',   # high-cardinality → neutral
}

FEATURE_LABELS = {
    'rain (mm)':         'Rainfall (mm)',
    'temp (C)':          'Temperature (°C)',
    'Year':              'Year',
    'pesticides_tonnes': 'Pesticides (t)',
    'Days_to_Harvest':   'Days to harvest',
    'Irrigation_Used':   'Irrigation used',
    'Fertilizer_Used':   'Fertilizer used',
    'Soil_Type':         'Soil type',
    'Area':              'Area (country)',
}

BOOL_COLORS   = {False: "#2D3D85", True: "#D93B67"}   # teal ramp
CAT_PALETTE   = ['#EF9F27', '#1D9E75', '#378ADD', '#D4537E', '#7F77DD']  # 5 Soil types


def _recover_feature_values(df_ohe: pd.DataFrame,
                             feat: str,
                             ohe_feature_names: list[str]) -> np.ndarray:
    """
    Reconstruct the original (pre-OHE) feature column from df_ohe.

    - Continuous / boolean : returned as-is (already a single column).
    - Categorical           : recovered via argmax over the OHE dummy columns.
    """
    ftype = FEATURE_TYPES[feat]

    if ftype in ('continuous', 'boolean'):
        return df_ohe[feat].to_numpy()

    # Categorical: find the OHE columns that belong to this feature
    ohe_cols = [c for c in ohe_feature_names if c.startswith(feat + '_')]
    if not ohe_cols:
        return np.full(len(df_ohe), np.nan)

    # argmax over the dummy columns → integer label → original category string
    sub = df_ohe[[c for c in ohe_cols if c in df_ohe.columns]]
    cat_idx  = np.argmax(sub.to_numpy(), axis=1)
    cat_names = [c.replace(feat + '_', '') for c in sub.columns]
    return np.array([cat_names[i] for i in cat_idx])


def _beeswarm_positions(shap_vals: np.ndarray,
                        y_center: float,
                        max_height: float = 0.35,
                        n_bins: int = 100) -> np.ndarray:
    """
    Pack dots into a beeswarm by binning along x and stacking vertically.
    Returns y coordinates for each dot.
    """
    y_out = np.full_like(shap_vals, y_center, dtype=float)
    bins  = np.linspace(shap_vals.min() - 1e-9,
                        shap_vals.max() + 1e-9, n_bins + 1)
    step  = max_height / max(1, len(shap_vals) // (n_bins * 2) + 1)

    for b in range(n_bins):
        mask = (shap_vals >= bins[b]) & (shap_vals < bins[b + 1])
        idx  = np.where(mask)[0]
        if len(idx) == 0:
            continue
        n = len(idx)
        # Centre-stack: 0, +step, -step, +2step, -2step …
        offsets = np.array([(k // 2 + 1) * step * (1 if k % 2 == 0 else -1)
                             if k > 0 else 0.0
                             for k in range(n)])
        offsets = np.clip(offsets, -max_height, max_height)
        y_out[idx] = y_center + offsets

    return y_out


def plot_shap_beeswarm(df_ohe: pd.DataFrame,
                       shap_values: dict[str, np.ndarray],
                       crop: str,
                       max_samples: int = 500,
                       figsize: tuple = (10, 7)) -> None:
    """
    SHAP beeswarm / summary plot for a single crop.

    Parameters
    ----------
    df_ohe       : full OHE dataframe (must contain 'item' column)
    shap_values  : dict  crop → raw shap matrix  (n_samples x n_ohe_features)
                   Tip: compute with  shap.TreeExplainer(model).shap_values(X)
    crop         : one of 'Maize', 'Rice', 'Wheat'
    max_samples  : subsample for speed / readability
    """
    # ── Subset to crop ────────────────────────────────────────────────────────
    df_crop = df_ohe[df_ohe['item'] == crop].drop(
        ['item', 'yield (t/ha)'], axis=1
    )
    ohe_feature_names = list(df_crop.columns)
    shap_mat = shap_values[crop]   # (n_samples, n_ohe_features)

    # Optional subsampling
    rng = np.random.default_rng(42)
    if len(df_crop) > max_samples:
        idx     = rng.choice(len(df_crop), max_samples, replace=False)
        df_crop = df_crop.iloc[idx].reset_index(drop=True)
        shap_mat = shap_mat[idx]

    # ── Aggregate OHE → original features ────────────────────────────────────
    agg = {}   # feat → {'shap': (n,), 'feat_vals': (n,), 'mean_abs': float}
    for feat, ohe_cols in FEATURE_GROUPS.items():
        col_idx  = [ohe_feature_names.index(c)
                    for c in ohe_cols if c in ohe_feature_names]
        grouped  = shap_mat[:, col_idx].sum(axis=1)
        feat_vals = _recover_feature_values(df_crop, feat, ohe_feature_names)
        agg[feat] = {
            'shap':      grouped,
            'feat_vals': feat_vals,
            'mean_abs':  float(np.mean(np.abs(grouped))),
        }

    # Sort features by mean |SHAP| ascending (bottom = least important)
    sorted_feats = sorted(agg, key=lambda f: agg[f]['mean_abs'])

    # ── Style ─────────────────────────────────────────────────────────────────
    plt.rcParams.update({
        'font.family':        'sans-serif',
        'axes.facecolor':     '#ffffff',
        'figure.facecolor':   '#ffffff',
        'axes.spines.top':    False,
        'axes.spines.right':  False,
        'axes.spines.left':   False,
        'axes.spines.bottom': True,
        'axes.edgecolor':     '#cccbc5',
        'axes.linewidth':     0.6,
        'xtick.color':        '#888780',
        'ytick.color':        '#444441',
        'grid.color':         '#e8e7e1',
        'grid.linewidth':     0.6,
    })

    fig, ax = plt.subplots(figsize=figsize)
    fig.subplots_adjust(left=0.22, right=0.82, top=0.88, bottom=0.1)

    legend_elements = []

    for y_idx, feat in enumerate(sorted_feats):
        shap_v    = agg[feat]['shap']
        feat_vals = agg[feat]['feat_vals']
        ftype     = FEATURE_TYPES[feat]
        y_pos     = _beeswarm_positions(shap_v, y_center=y_idx, n_bins=70)

        # ── Color by feature type ─────────────────────────────────────────────
        if ftype == 'continuous':
            cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", [BOOL_COLORS[False], BOOL_COLORS[True]])

            norm = mcolors.Normalize(vmin=np.nanpercentile(feat_vals, 2),
                                    vmax=np.nanpercentile(feat_vals, 98))

            colors = cmap(norm(feat_vals.astype(float)))

            ax.scatter(shap_v, y_pos, c=colors, s=7, alpha=0.6,
                       linewidths=0, rasterized=True)

        elif ftype == 'boolean':
            bool_vals = feat_vals.astype(bool)
            for val, col in BOOL_COLORS.items():
                mask = (bool_vals == val)
                ax.scatter(shap_v[mask], y_pos[mask], color=col,
                           s=7, alpha=0.7, linewidths=0,
                           label=str(val), rasterized=True)
            # Add to legend once only
            if y_idx == next(i for i, f in enumerate(sorted_feats)
                             if FEATURE_TYPES[f] == 'boolean'):
                for val, col in BOOL_COLORS.items():
                    legend_elements.append(
                        Line2D([0], [0], marker='o', color='none',
                               markerfacecolor=col, markersize=6,
                               label=f'{"True" if val else "False"}')
                    )

        elif ftype == 'categorical':
            cats = sorted(set(feat_vals))
            if len(cats) <= 8:
                cat_color_map = {c: CAT_PALETTE[i % len(CAT_PALETTE)]
                                 for i, c in enumerate(cats)}
                for cat, col in cat_color_map.items():
                    mask = (feat_vals == cat)
                    ax.scatter(shap_v[mask], y_pos[mask], color=col,
                               s=7, alpha=0.7, linewidths=0,
                               rasterized=True)
                if feat == 'Soil_Type':
                    for cat, col in cat_color_map.items():
                        legend_elements.append(
                            Line2D([0], [0], marker='o', color='none',
                                   markerfacecolor=col, markersize=6,
                                   label=cat)
                        )
            else:
                # High-cardinality (Area): neutral gray strip
                ax.scatter(shap_v, y_pos, color='#B4B2A9', s=7,
                           alpha=0.4, linewidths=0, rasterized=True)

                # ── Annotate top 4 highest + 1 lowest SHAP countries ──────────
                sorted_by_shap = np.argsort(shap_v)
                annotate_idx   = np.concatenate([
                    sorted_by_shap[-1:],   # top 1 highest, desc
                    sorted_by_shap[:1],           # single lowest
                ])

                already_labeled = set()
                text_objects    = []

                for rank, sample_idx in enumerate(annotate_idx):
                    country = feat_vals[sample_idx]

                    # Skip if this country was already annotated
                    if country in already_labeled:
                        continue
                    already_labeled.add(country)

                    x = shap_v[sample_idx]
                    y = y_pos[sample_idx]

                    is_lowest  = rank == len(annotate_idx) - 1
                    x_offset   = -18 if is_lowest else 18
                    ha         = 'right' if is_lowest else 'left'
                    dot_color  = BOOL_COLORS[True] if not is_lowest else BOOL_COLORS[False]  # purple / coral

                    # Highlighted dot
                    ax.scatter(x, y, color=dot_color, s=22,
                               zorder=5, linewidths=0)

                    # Annotation with a short connector
                    txt = ax.annotate(
                        country,
                        xy=(x, y),
                        xytext=(x_offset, 0),
                        textcoords='offset points',
                        fontsize=8.5,
                        color=dot_color,
                        va='center',
                        ha=ha,
                        arrowprops=dict(
                            arrowstyle='-',
                            color=dot_color,
                            lw=0.8,
                            shrinkA=0,
                            shrinkB=3,
                        ),
                        zorder=6,
                    )
                    text_objects.append(txt)

    # ── Axes ──────────────────────────────────────────────────────────────────
    ax.set_yticks(range(len(sorted_feats)))
    ax.set_yticklabels([FEATURE_LABELS[f] for f in sorted_feats],
                       fontsize=11, color='#444441')
    ax.tick_params(axis='y', length=0)
    ax.axvline(0, color='#aaa9a3', linewidth=0.8, zorder=3)
    ax.set_xlabel('SHAP value  (impact on predicted yield)',
                  fontsize=11, color='#888780', labelpad=8)
    ax.xaxis.grid(True, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', colors='#888780', labelsize=9.5)

    ax.set_title(f'Feature effect on yield - {crop}',
                 fontsize=13, color='#2C2C2A',
                 fontweight='normal', loc='left', pad=14)

    # ── Colorbars & legends ───────────────────────────────────────────────────
    # Continuous colorbar
    cbar_ax = fig.add_axes([0.84, 0.25, 0.02, 0.35])
    sm = cm.ScalarMappable(cmap=cmap,
                           norm=mcolors.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Low', 'High'], fontsize=9, color='#888780')
    cbar.outline.set_visible(False)
    cbar_ax.set_title('Feature\nvalue', fontsize=9,
                      color='#888780', pad=6, loc='left')

    # Categorical / boolean legend  
    if legend_elements:
        ax.legend(handles=legend_elements,
                  loc='lower right', fontsize=9,
                  frameon=False, labelcolor='#666560',
                  handlelength=1.0, ncol=2)

    plt.savefig(f'../assets/shap_beeswarm_{crop.lower()}.png',
                dpi=150, bbox_inches='tight', facecolor='#ffffff')

In [13]:
# ── Usage ──────────────────────────────────────────────────────────────────────
shap_matrices = {}
for crop, model_path in tqdm(MODELS_PATH.items(), desc="Computing SHAP value"):
    model = xgb.Booster(); model.load_model(model_path)
    df    = knn_df[knn_df['item'] == crop].drop(['item', 'yield (t/ha)'], axis=1)
    shap_matrices[crop] = shap.TreeExplainer(model).shap_values(df)

Computing SHAP value:   0%|          | 0/3 [00:00<?, ?it/s]

In [14]:
for crop in tqdm(['Maize', 'Rice', 'Wheat'], desc="Plotting"):
    plot_shap_beeswarm(knn_df, shap_matrices, crop)

Plotting:   0%|          | 0/3 [00:00<?, ?it/s]